In [9]:
import pandas as pd
import numpy
import os

In [32]:
lineage_file = pd.ExcelFile(r'C:\Users\AryanKumar\Downloads\Data Lineage\Order Fulfillment Dashboard_Upstream.xlsx')
lineage_sheet = lineage_file.parse('Order Fulfillment Dashboard_Ups')
lineage_sheet

,Name,Type,Business Name,Connector,Database,Schema,Lineage Depth,Description,Owner users,Owner groups,...,Announcement Message,Last Refresh on Source,Usage,Popularity,Qualified Name,Source Asset GUID,Impacted Asset GUID,Atlan URL,Immediate upstream,Immediate downstream
0,CDD Measures,Table,NaN,powerbi,NaN,NaN,1,NaN,NaN,NaN,...,NaN,NaN,NaN,NO USAGE,default/powerbi/1715713576/c2fdcbf8-d5b6-475d-...,d586a708-8896-4eb8-923a-b79c3311bbf6,99274617-ce04-4186-b92b-094b1c2b30b8,https://employbridge.atlan.com/assets/99274617...,NaN,Order Fulfillment Dashboard (default/powerbi/1...
1,ACCOUNTBASE,Table,NaN,powerbi,NaN,NaN,1,NaN,NaN,NaN,...,NaN,NaN,NaN,NO USAGE,default/powerbi/1715713576/c2fdcbf8-d5b6-475d-...,d586a708-8896-4eb8-923a-b79c3311bbf6,dc3acfff-ca35-49b5-ba63-ebae1f9da1c5,https://employbridge.atlan.com/assets/dc3acfff...,ACCOUNTBASE (default/snowflake/1715075875/PROD...,Order Fulfillment Dashboard (default/powerbi/1...
2,BRANCH_DIM,Table,NaN,powerbi,NaN,NaN,1,NaN,NaN,NaN,...,NaN,NaN,NaN,NO USAGE,default/powerbi/1715713576/c2fdcbf8-d5b6-475d-...,d586a708-8896-4eb8-923a-b79c3311bbf6,9dd39b50-c638-4fc8-99f8-5f85cf388a80,https://employbridge.atlan.com/assets/9dd39b50...,BRANCH_DIM (default/snowflake/1715075875/PROD_...,Order Fulfillment Dashboard (default/powerbi/1...
3,NPG_VIP_DL_LDG_CLIENT_MASTER,Table,NaN,powerbi,NaN,NaN,1,NaN,NaN,NaN,...,NaN,NaN,NaN,NO USAGE,default/powerbi/1715713576/c2fdcbf8-d5b6-475d-...,d586a708-8896-4eb8-923a-b79c3311bbf6,bfe9df04-c7e4-4938-9099-350851cebb17,https://employbridge.atlan.com/assets/bfe9df04...,NPG_VIP_DL_LDG_CLIENT_MASTER (default/snowflak...,Order Fulfillment Dashboard (default/powerbi/1...
4,TBL_WBS_RECAST,Table,NaN,powerbi,NaN,NaN,1,NaN,NaN,NaN,...,NaN,NaN,NaN,NO USAGE,default/powerbi/1715713576/c2fdcbf8-d5b6-475d-...,d586a708-8896-4eb8-923a-b79c3311bbf6,9df82820-8f3e-43dd-86b0-de22b3494571,https://employbridge.atlan.com/assets/9df82820...,TBL_WBS_RECAST (default/snowflake/1715075875/D...,Order Fulfillment Dashboard (default/powerbi/1...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
325,ACCOUNT_AUDITBASE_TEMP1,Table,NaN,snowflake,DATALAKE,PUBLIC,6,NaN,NaN,NaN,...,NaN,"Mar 7, 2026, 4:21:37 PM",NaN,NO USAGE,default/snowflake/1715075875/DATALAKE/PUBLIC/A...,d586a708-8896-4eb8-923a-b79c3311bbf6,88e0ea9a-7ab4-4548-b3e7-66e2c89c137e,https://employbridge.atlan.com/assets/88e0ea9a...,DATALAKE_AUDITBASE (default/snowflake/17150758...,ACCOUNT_AUDITBASE_TEMP2 (default/snowflake/171...
326,AUDITBASE,Table,NaN,snowflake,PROD_DATALAKE,CRM_MSCRM,6,NaN,NaN,NaN,...,NaN,"Mar 7, 2026, 9:45:18 PM",NaN,NO USAGE,default/snowflake/1715075875/PROD_DATALAKE/CRM...,d586a708-8896-4eb8-923a-b79c3311bbf6,45374d64-f5f8-4d48-a538-835ffa415ade,https://employbridge.atlan.com/assets/45374d64...,attrep_changesD089CE7E1DCCE970 (default/snowfl...,OE_EMPLOYEE (default/snowflake/1715075875/SAND...
327,OE_EMPLOYEEAUDITHIST,View,NaN,snowflake,PROD_ENTERPRISE,OPERATIONS,6,NaN,NaN,NaN,...,NaN,"Sep 17, 2021, 1:15:23 AM",NaN,NO USAGE,default/snowflake/1715075875/PROD_ENTERPRISE/O...,d586a708-8896-4eb8-923a-b79c3311bbf6,cc6c8195-99c2-4735-91a2-968d8c8bec36,https://employbridge.atlan.com/assets/cc6c8195...,DATALAKE_AUDITBASE (default/snowflake/17150758...,OE_EMPLOYEE (default/snowflake/1715075875/SAND...
328,LDG_PERSON_DETAILS,Table,NaN,snowflake,PROD_DATALAKE,UKG,6,NaN,NaN,NaN,...,NaN,"Mar 7, 2026, 2:32:45 PM",NaN,NO USAGE,default/snowflake/1715075875/PROD_DATALAKE/UKG...,d586a708-8896-4eb8-923a-b79c3311bbf6,cba09e9e-8b88-4000-9cd6-804f17e45bc2,https://employbridge.atlan.com/assets/cba09e9e...,NaN,PERSON_DETAILS (default/snowflake/1715075875/P...


In [52]:
def clean_path(s):
    start = s.find("(") + 1
    end = s.find(")")
    inside = s[start:end]
    return inside

In [110]:
df = lineage_sheet[['Name','Type','Connector','Database','Schema','Lineage Depth','Immediate upstream']].copy()
queries = df[(df['Connector'] == 'powerbi') & (df['Immediate upstream'].notnull())][['Name','Immediate upstream']].copy()
queries.rename(columns={'Name': 'Query', 'Immediate upstream': 'L1'}, inplace=True)
queries.index = pd.RangeIndex(1, len(queries)+1)
queries['L1'] = (
    queries['L1']
        .str.extract(r'\((.*?)\)')[0]
        .str.split('/', expand=True)
        .iloc[:, 3:6]
        .apply(lambda row: '.'.join(row.dropna()), axis=1)
)
queries[['Database_L1', 'Schema_L1', 'Object_L1']] = queries['L1'].str.split('.', expand=True)
queries

,Query,L1,Database_L1,Schema_L1,Object_L1
1,ACCOUNTBASE,PROD_DATALAKE.CRM_MSCRM.ACCOUNTBASE,PROD_DATALAKE,CRM_MSCRM,ACCOUNTBASE
2,BRANCH_DIM,PROD_EDW.ENT.BRANCH_DIM,PROD_EDW,ENT,BRANCH_DIM
3,NPG_VIP_DL_LDG_CLIENT_MASTER,PROD_NPG_VIP.LDG.NPG_VIP_DL_LDG_CLIENT_MASTER,PROD_NPG_VIP,LDG,NPG_VIP_DL_LDG_CLIENT_MASTER
4,TBL_WBS_RECAST,DEV_SANDBOX_MERILYTICS.DATAWAREHOUSE.TBL_WBS_R...,DEV_SANDBOX_MERILYTICS,DATAWAREHOUSE,TBL_WBS_RECAST
5,VW_MMM_CHURN_CATEGORY_WBS,DEV_SANDBOX_MERILYTICS.DATAWAREHOUSE.VW_MMM_CH...,DEV_SANDBOX_MERILYTICS,DATAWAREHOUSE,VW_MMM_CHURN_CATEGORY_WBS
6,VW_ACCOUNTINGUNIT_ZIPCODE,DEV_SANDBOX_MERILYTICS.DATAWAREHOUSE.VW_ACCOUN...,DEV_SANDBOX_MERILYTICS,DATAWAREHOUSE,VW_ACCOUNTINGUNIT_ZIPCODE
7,OE_JOBORDER_FILL_HISTORY_ALL,PROD_ENTERPRISE.OPERATIONS.OE_JOBORDER_FILL_HI...,PROD_ENTERPRISE,OPERATIONS,OE_JOBORDER_FILL_HISTORY_ALL
8,VW_USERBRANCH,PROD_EDW.ENT.VW_USERBRANCH,PROD_EDW,ENT,VW_USERBRANCH
9,CDA_REPLACEMENTORDERTRACKINGBASE,PROD_DATALAKE.CRM_MSCRM.CDA_REPLACEMENTORDERTR...,PROD_DATALAKE,CRM_MSCRM,CDA_REPLACEMENTORDERTRACKINGBASE
10,VW_PRODUCER_LEVEL_FUNNEL,PROD_ENTERPRISE.ENT.VW_PRODUCER_LEVEL_FUNNEL,PROD_ENTERPRISE,ENT,VW_PRODUCER_LEVEL_FUNNEL


In [113]:
queries_merged = queries.copy()
current_level = 2
count = 0
while count < 6:  # Arbitrary limit to prevent infinite loops
    merge_keys = [f'Database_L{current_level-1}', f'Schema_L{current_level-1}', f'Object_L{current_level-1}']
    queries_merged = pd.merge(
        queries_merged,
        df[['Name','Database','Schema','Immediate upstream']],
        left_on=merge_keys,
        right_on=['Database', 'Schema', 'Name'],
        how='left'
    )
    queries_merged.rename(columns={'Database': f'Database_L{current_level}', 'Schema': f'Schema_L{current_level}', 'Name': f'Object_L{current_level}', 'Immediate upstream': f'L{current_level}'}, inplace=True)
    
    queries_merged[f'L{current_level}'] = queries_merged[f'L{current_level}'].str.split(',')
    queries_merged = queries_merged.explode(f'L{current_level}')
    
    queries_merged.index = pd.RangeIndex(1, len(queries_merged)+1)
    
    queries_merged[f'L{current_level}'] = (
        queries_merged[f'L{current_level}']
            .str.extract(r'\((.*?)\)')[0]
            .str.split('/', expand=True)
            .iloc[:, 3:6]
            .apply(lambda row: '.'.join(row.dropna()), axis=1)
    )
    
    queries_merged[[f'Database_L{current_level}', f'Schema_L{current_level}', f'Object_L{current_level}']] = queries_merged[f'L{current_level}'].str.split('.', expand=True)
    count += 1
    if queries_merged[f'L{current_level}'].isna().all():
        # Delete the empty column
        queries_merged.drop(columns=[f'L{current_level}', f'Database_L{current_level}', f'Schema_L{current_level}', f'Object_L{current_level}'], inplace=True)
        break
    current_level += 1
queries_merged

,Query,L1,Database_L1,Schema_L1,Object_L1,Object_L2,Database_L2,Schema_L2,L2,Object_L3,...,Schema_L5,L5,Object_L6,Database_L6,Schema_L6,L6,Object_L7,Database_L7,Schema_L7,L7
1,ACCOUNTBASE,PROD_DATALAKE.CRM_MSCRM.ACCOUNTBASE,PROD_DATALAKE,CRM_MSCRM,ACCOUNTBASE,attrep_changes3F84AF30F8F36C10,PROD_DATALAKE,CRM_MSCRM,PROD_DATALAKE.CRM_MSCRM.attrep_changes3F84AF30...,None,...,None,,None,,None,,None,,None,
2,BRANCH_DIM,PROD_EDW.ENT.BRANCH_DIM,PROD_EDW,ENT,BRANCH_DIM,STRINGMAPBASE,PROD_DATALAKE,CRM_MSCRM,PROD_DATALAKE.CRM_MSCRM.STRINGMAPBASE,STRINGMAPBASE,...,CRM_MSCRM,PROD_DATALAKE.CRM_MSCRM.STRINGMAPBASE,STRINGMAPBASE,PROD_DATALAKE,CRM_MSCRM,PROD_DATALAKE.CRM_MSCRM.STRINGMAPBASE,STRINGMAPBASE,PROD_DATALAKE,CRM_MSCRM,PROD_DATALAKE.CRM_MSCRM.STRINGMAPBASE
3,BRANCH_DIM,PROD_EDW.ENT.BRANCH_DIM,PROD_EDW,ENT,BRANCH_DIM,STRINGMAPBASE,PROD_DATALAKE,CRM_MSCRM,PROD_DATALAKE.CRM_MSCRM.STRINGMAPBASE,STRINGMAPBASE,...,CRM_MSCRM,PROD_DATALAKE.CRM_MSCRM.STRINGMAPBASE,STRINGMAPBASE,PROD_DATALAKE,CRM_MSCRM,PROD_DATALAKE.CRM_MSCRM.STRINGMAPBASE,attrep_changes3F84AF30F8F36C10,PROD_DATALAKE,CRM_MSCRM,PROD_DATALAKE.CRM_MSCRM.attrep_changes3F84AF30...
4,BRANCH_DIM,PROD_EDW.ENT.BRANCH_DIM,PROD_EDW,ENT,BRANCH_DIM,STRINGMAPBASE,PROD_DATALAKE,CRM_MSCRM,PROD_DATALAKE.CRM_MSCRM.STRINGMAPBASE,STRINGMAPBASE,...,CRM_MSCRM,PROD_DATALAKE.CRM_MSCRM.STRINGMAPBASE,attrep_changes3F84AF30F8F36C10,PROD_DATALAKE,CRM_MSCRM,PROD_DATALAKE.CRM_MSCRM.attrep_changes3F84AF30...,None,,None,
5,BRANCH_DIM,PROD_EDW.ENT.BRANCH_DIM,PROD_EDW,ENT,BRANCH_DIM,STRINGMAPBASE,PROD_DATALAKE,CRM_MSCRM,PROD_DATALAKE.CRM_MSCRM.STRINGMAPBASE,STRINGMAPBASE,...,CRM_MSCRM,PROD_DATALAKE.CRM_MSCRM.attrep_changes3F84AF30...,None,,None,,None,,None,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8958,VW_JOBORDER_DETAIL_PRICING_ANALYTICS_NAV,DEV_SANDBOX_MERILYTICS.DATAWAREHOUSE.VW_JOBORD...,DEV_SANDBOX_MERILYTICS,DATAWAREHOUSE,VW_JOBORDER_DETAIL_PRICING_ANALYTICS_NAV,VW_PRICING_QUERY_REPORT_FPR,DEV_SANDBOX_MERILYTICS,DATAWAREHOUSE,DEV_SANDBOX_MERILYTICS.DATAWAREHOUSE.VW_PRICIN...,REPORT_FPR,...,PUBLIC,DATALAKE.PUBLIC.DIM_ASSIGNMENT,CONTACTBASE,PROD_DATALAKE,CRM_MSCRM,PROD_DATALAKE.CRM_MSCRM.CONTACTBASE,attrep_changes3F84AF30F8F36C10,PROD_DATALAKE,CRM_MSCRM,PROD_DATALAKE.CRM_MSCRM.attrep_changes3F84AF30...
8959,VW_JOBORDER_DETAIL_PRICING_ANALYTICS_NAV,DEV_SANDBOX_MERILYTICS.DATAWAREHOUSE.VW_JOBORD...,DEV_SANDBOX_MERILYTICS,DATAWAREHOUSE,VW_JOBORDER_DETAIL_PRICING_ANALYTICS_NAV,VW_PRICING_QUERY_REPORT_FPR,DEV_SANDBOX_MERILYTICS,DATAWAREHOUSE,DEV_SANDBOX_MERILYTICS.DATAWAREHOUSE.VW_PRICIN...,REPORT_FPR,...,PUBLIC,DATALAKE.PUBLIC.DIM_ASSIGNMENT,DIM_ASSIGNMENT,DATALAKE,PUBLIC,DATALAKE.PUBLIC.DIM_ASSIGNMENT,CDA_ASSIGNMENTBASE,PROD_DATALAKE,CRM_MSCRM,PROD_DATALAKE.CRM_MSCRM.CDA_ASSIGNMENTBASE
8960,VW_JOBORDER_DETAIL_PRICING_ANALYTICS_NAV,DEV_SANDBOX_MERILYTICS.DATAWAREHOUSE.VW_JOBORD...,DEV_SANDBOX_MERILYTICS,DATAWAREHOUSE,VW_JOBORDER_DETAIL_PRICING_ANALYTICS_NAV,VW_PRICING_QUERY_REPORT_FPR,DEV_SANDBOX_MERILYTICS,DATAWAREHOUSE,DEV_SANDBOX_MERILYTICS.DATAWAREHOUSE.VW_PRICIN...,REPORT_FPR,...,PUBLIC,DATALAKE.PUBLIC.DIM_ASSIGNMENT,DIM_ASSIGNMENT,DATALAKE,PUBLIC,DATALAKE.PUBLIC.DIM_ASSIGNMENT,SYSTEMUSERBASE,PROD_DATALAKE,CRM_MSCRM,PROD_DATALAKE.CRM_MSCRM.SYSTEMUSERBASE
8961,VW_JOBORDER_DETAIL_PRICING_ANALYTICS_NAV,DEV_SANDBOX_MERILYTICS.DATAWAREHOUSE.VW_JOBORD...,DEV_SANDBOX_MERILYTICS,DATAWAREHOUSE,VW_JOBORDER_DETAIL_PRICING_ANALYTICS_NAV,VW_PRICING_QUERY_REPORT_FPR,DEV_SANDBOX_MERILYTICS,DATAWAREHOUSE,DEV_SANDBOX_MERILYTICS.DATAWAREHOUSE.VW_PRICIN...,REPORT_FPR,...,PUBLIC,DATALAKE.PUBLIC.DIM_ASSIGNMENT,DIM_ASSIGNMENT,DATALAKE,PUBLIC,DATALAKE.PUBLIC.DIM_ASSIGNMENT,CONTACTBASE,PROD_DATALAKE,CRM_MSCRM,PROD_DATALAKE.CRM_MSCRM.CONTACTBASE
